# Process EMG recordings with bioread

APP, PsychoPy and slow-rate files → FIR → comb → RMS → %MVC → four task segments.

No custom functions or helper scripts. Hidden channels are read automatically. **Filter defaults below are a candidate; exact AcqKnowledge equivalence is unverified.**

Dependencies: `bioread numpy scipy pandas h5py` (see `requirements.txt`). Paths are relative to the notebook working directory.

### Imports

In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import bioread
import h5py
import numpy as np
import pandas as pd
from scipy import signal

### Input files

In [ ]:
participant = "P001"
files = {
    "APP": Path("input/P001_APP_raw.acq"),
    "PsychoPy": Path("input/P001_PsychoPy_raw.acq"),
    "SlowRate": Path("input/P001_SlowRate_raw.acq"),
}
output_directory = Path("output/P001")

### Read the ACQ files

In [ ]:
recordings = {}
for task, path in files.items():
    recordings[task] = bioread.read_file(str(path))

### List all channels, including hidden ones

In [ ]:
channel_rows = []
for task, recording in recordings.items():
    for index, channel in enumerate(recording.channels):
        channel_rows.append({"task": task, "index": index, "order_num": channel.order_num,
                             "name": channel.name, "units": channel.units,
                             "rate_hz": channel.samples_per_second, "samples": len(channel.data)})
pd.DataFrame(channel_rows)

### Choose the two raw channels

Enter the **index** from the table for each original raw channel 3/4. File-list indices are not necessarily AcqKnowledge channel numbers. Confirm which is ZM and which is CS.

In [ ]:
channel_indices = {
    "APP": {"ZM": None, "CS": None},
    "PsychoPy": {"ZM": None, "CS": None},
    "SlowRate": {"ZM": None, "CS": None},
}

### Select ZM and CS

In [ ]:
selected = {}
for task, indices in channel_indices.items():
    assert all(type(i) is int and 0 <= i < len(recordings[task].channels) for i in indices.values()), task
    assert indices["ZM"] != indices["CS"], "ZM and CS must be different channels"
    for muscle, index in indices.items():
        selected[task, muscle] = recordings[task].channels[index]

### Get scaled raw samples

In [ ]:
raw = {key: np.asarray(channel.data, dtype=float).copy() for key, channel in selected.items()}
rates = {task: float(selected[task, "ZM"].samples_per_second) for task in files}
units = {key: channel.units for key, channel in selected.items()}

### Check samples and rates

In [ ]:
for task in files:
    assert np.isfinite(rates[task]) and rates[task] > 1000, "500 Hz cutoff needs rate > 1000 Hz"
    assert selected[task, "CS"].samples_per_second == rates[task], "Channel rates differ"
    assert len(raw[task, "ZM"]) == len(raw[task, "CS"]), "Channel lengths differ"
for values in raw.values():
    assert values.ndim == 1 and len(values) >= 1000 and np.isfinite(values).all()

### Participant MVC values

Use this participant’s baseline MVC values, in the same units as the raw signals.

In [ ]:
mvc = {"ZM": None, "CS": None}
mvc_units = {"ZM": "mV", "CS": "mV"}

### Check MVC values and units

In [ ]:
for muscle, value in mvc.items():
    assert value is not None and np.isfinite(value) and value > 0, f"Enter a positive {muscle} MVC"
for task, muscle in selected:
    assert units[task, muscle] == mvc_units[muscle], f"Unit mismatch: {task}, {muscle}"

### FIR settings

301 taps/Bartlett/centered convolution are explicit assumptions, not verified AcqKnowledge defaults.

In [ ]:
low_hz, high_hz = 28, 500
numtaps = 301
fir_window = "bartlett"
fir_scale = True
fir_padding = "reflect"

### Calculate FIR coefficients

In [ ]:
assert type(numtaps) is int and numtaps >= 3 and numtaps % 2 == 1
assert 0 < low_hz < high_hz < min(rates.values()) / 2
fir_coefficients = {}
for task, rate in rates.items():
    fir_coefficients[task] = signal.firwin(numtaps, [low_hz, high_hz], fs=rate,
                                         pass_zero=False, window=fir_window, scale=fir_scale)

### Apply the band-pass filter

In [ ]:
bandpass = {}
half_width = numtaps // 2
for key, values in raw.items():
    padded = np.pad(values, (half_width, half_width), mode=fir_padding)
    bandpass[key] = signal.convolve(padded, fir_coefficients[key[0]], mode="valid")

### Comb filter settings

`quality_factor` controls notch width. `harmonics` is a positive count or `"all"` below Nyquist. This notebook applies each notch once, forward, with zero initial state.

In [ ]:
line_hz = 60
quality_factor = 5
harmonics = "all"

### Calculate the notch coefficients

In [ ]:
assert np.isfinite(line_hz) and line_hz > 0 and np.isfinite(quality_factor) and quality_factor > 0
notches = {}
for task, rate in rates.items():
    maximum = int(np.ceil(rate / (2 * line_hz))) - 1
    count = maximum if harmonics == "all" else harmonics
    assert type(count) is int and 1 <= count <= maximum
    notches[task] = [signal.iirnotch(line_hz * k, quality_factor, fs=rate)
                     for k in range(1, count + 1)]

### Apply the comb filter

In [ ]:
comb = {key: values.copy() for key, values in bandpass.items()}
for key in comb:
    for numerator, denominator in notches[key[0]]:
        comb[key] = signal.lfilter(numerator, denominator, comb[key])

### RMS settings

In [ ]:
rms_samples = 1000
rms_alignment = "centered"
rms_padding = "reflect"

### Square the filtered samples

In [ ]:
squared = {key: values ** 2 for key, values in comb.items()}

### Calculate the moving mean

A centered 1,000-sample window uses i−500 through i+499. Padding keeps the original length.

In [ ]:
assert type(rms_samples) is int and 1 <= rms_samples <= min(map(len, squared.values()))
assert rms_alignment in {"centered", "trailing"}
left = rms_samples // 2 if rms_alignment == "centered" else rms_samples - 1
right = rms_samples - 1 - left
mean_square = {}
for key, values in squared.items():
    padded = np.pad(values, (left, right), mode=rms_padding)
    mean_square[key] = signal.convolve(padded, np.ones(rms_samples) / rms_samples,
                                       mode="valid", method="direct")

### Take the square root

In [ ]:
rms = {key: np.sqrt(np.maximum(values, 0)) for key, values in mean_square.items()}

### Normalize to percent MVC

In [ ]:
percent_mvc = {key: 100 * values / mvc[key[1]] for key, values in rms.items()}

### Check the processed signals

In [ ]:
for key in raw:
    for stage in (bandpass, comb, rms, percent_mvc):
        assert len(stage[key]) == len(raw[key]) and np.isfinite(stage[key]).all()
    np.testing.assert_allclose(percent_mvc[key] * mvc[key[1]] / 100, rms[key])

### List the event markers

`bioread` provides marker times in seconds; its marker sample indices use the file’s base sample rate.

In [ ]:
marker_rows = []
for task, recording in recordings.items():
    for index, marker in enumerate(recording.event_markers or []):
        marker_rows.append({"task": task, "index": index, "text": marker.text,
                            "time_s": float(marker.time_index), "type": marker.type_code,
                            "source_channel": marker.channel_number})
pd.DataFrame(marker_rows)

### Read the source journals

Preview only; complete journals are saved. Check for timing exceptions before choosing markers.

In [ ]:
journals = {task: str(recording.journal or "") for task, recording in recordings.items()}
for task, journal in journals.items():
    print(task, "\n", journal[:1500], "\n")

### Choose the cue and star markers

Use the marker **index** from the table. Selecting an explicit index avoids ambiguous repeated labels.

In [ ]:
marker_indices = {"video_cue": None, "interview_star": None,
                  "psychopy_star": None, "slowrate_star": None}
marker_tasks = {"video_cue": "APP", "interview_star": "APP",
                "psychopy_star": "PsychoPy", "slowrate_star": "SlowRate"}

### Get the selected marker times

In [ ]:
event_times = {}
for event, index in marker_indices.items():
    events = recordings[marker_tasks[event]].event_markers or []
    assert type(index) is int and 0 <= index < len(events), f"Choose the {event} marker"
    event_times[event] = float(events[index].time_index)
    assert np.isfinite(event_times[event]) and event_times[event] >= 0
pd.Series(event_times, name="seconds")

### Define the seven output recordings

In [ ]:
segments = {
    "APP_filter": ("APP", 0, None),
    "PsychoPy_filter": ("PsychoPy", 0, None),
    "SlowRate_filter": ("SlowRate", 0, None),
    "1_video_filter_edit": ("APP", event_times["video_cue"], 120),
    "2_interview_filter_edit": ("APP", event_times["interview_star"], None),
    "PsychoPy_filter_edit": ("PsychoPy", event_times["psychopy_star"], None),
    "SlowRate_filter_edit": ("SlowRate", event_times["slowrate_star"], None),
}

### Convert boundaries to sample indices

Nearest sample, half up; end is exclusive. Filtering has already happened on the full recordings.

In [ ]:
bounds = {}
for name, (task, start_seconds, duration) in segments.items():
    start = int(np.floor(start_seconds * rates[task] + 0.5))
    stop = len(raw[task, "ZM"]) if duration is None else start + int(np.floor(duration * rates[task] + 0.5))
    assert 0 <= start < stop <= len(raw[task, "ZM"]), f"Invalid crop: {name}"
    bounds[name] = (start, stop)
pd.DataFrame(bounds, index=["start_sample", "stop_exclusive"]).T

### Crop each processing stage

In [ ]:
stages = {"raw": raw, "bandpass": bandpass, "comb": comb, "rms": rms, "pct_mvc": percent_mvc}
outputs = {}
for name, (task, _, _) in segments.items():
    start, stop = bounds[name]
    outputs[name] = {f"{muscle}_{stage}": values[task, muscle][start:stop]
                     for stage, values in stages.items() for muscle in ("ZM", "CS")}

### Shift retained event times

In [ ]:
output_markers = {}
for name, (task, _, _) in segments.items():
    start, stop = bounds[name]
    output_markers[name] = [{**row, "original_time_s": row["time_s"],
                             "time_s": row["time_s"] - start / rates[task]}
                            for row in marker_rows if row["task"] == task
                            and start / rates[task] <= row["time_s"] < stop / rates[task]]

### Record the processing settings

In [ ]:
processing = {
    "fir": {"low_hz": low_hz, "high_hz": high_hz, "numtaps": numtaps,
            "window": fir_window, "scale": fir_scale, "application": "centered", "padding": fir_padding},
    "comb": {"base_hz": line_hz, "q": quality_factor, "harmonics": harmonics,
             "application": "forward", "initial": "zero"},
    "rms": {"samples": rms_samples, "alignment": rms_alignment, "padding": rms_padding},
}
versions = {name: version(name) for name in ("bioread", "numpy", "scipy", "pandas", "h5py")}

### Record source file hashes

In [ ]:
source_hashes = {}
for task, path in files.items():
    with path.open("rb") as source:
        source_hashes[task] = hashlib.file_digest(source, "sha256").hexdigest()

### Assemble the output metadata

Journal text remains unchanged and may contain original timestamps. Marker channel IDs still refer to the source file.

In [ ]:
metadata = {}
for name, (task, _, _) in segments.items():
    start, stop = bounds[name]
    metadata[name] = {"source_file": str(files[task].resolve()), "source_sha256": source_hashes[task],
        "participant": participant, "task": task, "processing": processing, "versions": versions,
        "mvc": mvc, "mvc_units": mvc_units, "selected_indices": channel_indices[task],
        "source_channels": [row for row in channel_rows if row["task"] == task],
        "source_journal": journals[task], "source_markers": [row for row in marker_rows if row["task"] == task],
        "markers": output_markers[name], "original_start_s": start / rates[task],
        "crop_samples": [start, stop], "fir_coefficients": fir_coefficients[task].tolist(),
        "notch_coefficients": [{"b": b.tolist(), "a": a.tolist()} for b, a in notches[task]],
        "equivalence": "Candidate; AcqKnowledge equivalence unverified"}

### Choose new output filenames

In [ ]:
assert participant and all(c.isalnum() or c in "_-" for c in participant)
output_directory.mkdir(parents=True, exist_ok=True)
output_paths = {name: output_directory / f"{participant}_{name}" for name in outputs}
for stem in output_paths.values():
    for suffix in (".h5", ".txt", ".json"):
        assert not stem.with_suffix(suffix).exists(), f"File already exists: {stem}{suffix}"

### Save HDF5 signals

In [ ]:
for name, arrays in outputs.items():
    task = segments[name][0]
    with h5py.File(output_paths[name].with_suffix(".h5"), "x") as saved:
        saved.attrs["format"] = "acq-emg-pipeline-v1"
        saved.create_dataset("metadata_json", data=json.dumps(metadata[name], allow_nan=False))
        group = saved.create_group("signals")
        for key, values in arrays.items():
            dataset = group.create_dataset(key, data=values, compression="gzip", shuffle=True)
            dataset.attrs.update(sample_rate_hz=rates[task], label=key,
                units="%" if key.endswith("pct_mvc") else units[task, key.split("_")[0]],
                origin_s=bounds[name][0] / rates[task])

### Save four-channel text exports

No header or time column. Order: ZM RMS, CS RMS, ZM %MVC, CS %MVC.

In [ ]:
export_columns = ["ZM_rms", "CS_rms", "ZM_pct_mvc", "CS_pct_mvc"]
for name, arrays in outputs.items():
    with output_paths[name].with_suffix(".txt").open("x") as saved:
        np.savetxt(saved, np.column_stack([arrays[key] for key in export_columns]),
                   delimiter="\t", fmt="%.17g")

### Save text-export metadata

In [ ]:
for name in outputs:
    task = segments[name][0]
    sidecar = {**metadata[name], "columns": export_columns, "sample_interval_s": 1 / rates[task],
               "column_units": [units[task, "ZM"], units[task, "CS"], "%", "%"]}
    with output_paths[name].with_suffix(".json").open("x") as saved:
        json.dump(sidecar, saved, indent=2, allow_nan=False)

### List saved results

In [ ]:
pd.DataFrame([{"file": str(stem.with_suffix(".h5")),
               "seconds": len(outputs[name]["ZM_rms"]) / rates[segments[name][0]]}
              for name, stem in output_paths.items()])

[bioread](https://github.com/uwmadison-chm/bioread) · [SciPy FIR](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html) · [SciPy notch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.iirnotch.html) · [BIOPAC guide](https://www.biopac.com/wp-content/uploads/AcqKnowledge-5-Software-Guide.pdf)